# LangGraph 노트북 멀티턴 챗봇 데모

이 노트북은 상위 폴더의 `chatbot.py` 를 셀 단위로 사용해봅니다.

- **폐쇄망 친화**: MockLLM + `MemorySaver` — 외부 네트워크 호출, 바이너리 파일 저장 없음
- **관찰성(observability)**: 노드 / LLM / tool 계층형 span + 토큰 수 + latency
- **반출 편의**: 트레이스는 외부 fetch 없는 self-contained HTML 한 파일로 저장

> 주의: `MockLLM` 은 에코 스타일 시뮬레이터입니다. 실제 응답 품질이 아니라 *워크플로/트레이스 구조* 를 보기 위한 것입니다.

## 0. 인터랙티브 채팅 UI — 셀 Output 안에서 멀티턴 대화

아래 셀을 실행하면 셀 output 영역에 **입력창 + 보내기 / 트레이스 / 리셋 버튼 + 대화 풍선** 이 렌더됩니다. 매 턴마다 `bot.chat("...")` 을 수동으로 호출할 필요 없이, 위젯 안에서 바로 멀티턴 대화가 가능합니다.

- 필요 패키지: `ipywidgets` (JupyterLab 에 기본 포함되는 경우가 많음)
- **보내기** — 현재 입력을 한 턴으로 전송하고 응답을 아래에 추가
- **트레이스 보기** — 지금까지의 span 트리(토큰/latency 포함)를 셀 안에 렌더
- **새 대화** — `bot.reset()`, 새 `thread_id` 로 대화 맥락을 끊음 (Tracer는 유지)

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from chatbot import Chatbot

chat_bot = Chatbot()
chat_bot.chat_ui()   # 이 셀의 output 안에서 멀티턴 대화가 이뤄집니다

---
## (참고) 아래는 같은 챗봇을 셀 단위로 단계별로 보여주는 예시입니다.

## 1. 셋업 — 챗봇 생성

In [2]:
import os, sys
# chatbot.py 는 examples/ 의 상위 디렉토리에 있습니다.
sys.path.insert(0, os.path.abspath(".."))

from chatbot import Chatbot

bot = Chatbot()            # MockLLM + 새 Tracer + 랜덤 thread_id 로 초기화
print("thread_id:", bot.thread_id)

thread_id: d28341a0


## 2. 첫 번째 턴 — 평범한 대화

`bot.chat(...)` 는 한 턴의 최종 assistant 응답 문자열을 돌려줍니다. 내부적으로는 `langgraph` 그래프(`chat` 노드)가 실행됩니다.

In [3]:
print(bot.chat("안녕하세요, 자기소개 부탁드립니다."))

[턴 1] 입력: '안녕하세요, 자기소개 부탁드립니다.'. 대화에는 지금까지 1개 메시지가 누적되어 있습니다.


## 3. 두 번째 턴 — tool 호출 경로 시연

입력에 "계산 / 더하기 / +" 등의 키워드와 숫자가 있으면, 그래프가 `tool:calculator` span 을 먼저 호출한 뒤 그 결과를 LLM에 전달합니다. 트레이스에서 `LLM → tool → LLM` 계층을 곧이어 확인해봅니다.

In [4]:
print(bot.chat("12 + 7 + 100 을 계산해줘. 결과만 한 문장으로."))

[턴 2] 도구 실행 결과를 확인했습니다. → 계산기 결과: 119 (입력 숫자: [12, 7, 100]) (원 질문: '12 + 7 + 100 을 계산해줘. 결과만 한 문장으로.')


## 4. 세 번째 턴 — 이전 맥락이 유지되는지 확인

같은 `thread_id` 에서 `bot.chat` 을 반복하면 `MemorySaver` 가 이전 메시지들을 불러와 state에 합칩니다. LLM에 전달되는 입력 토큰 수가 점점 늘어나는 것으로도 확인할 수 있습니다.

In [5]:
print(bot.chat("방금 내가 어떤 질문을 두 개 했는지 요약해줘."))

[턴 3] 입력: '방금 내가 어떤 질문을 두 개 했는지 요약해줘.'. 대화에는 지금까지 6개 메시지가 누적되어 있습니다.


## 5. 대화 이력 시각화 — 채팅 풍선

`show_history()` 는 IPython.display 로 HTML 풍선 UI 를 셀 안에 렌더합니다.

In [5]:
bot.show_history()

## 6. 트레이스(observability) 뷰어를 셀 안에 표시

- 상단에 총 span 수 / LLM 호출 수 / 입·출력 토큰 합계 / 최상위 누적 latency 가 요약됩니다.
- 각 span 헤더를 클릭하면 inputs/outputs JSON 상세가 펼쳐집니다.
- 색상: <span style="color:#6366f1"><b>LLM</b></span> · <span style="color:#10b981"><b>tool</b></span> · <span style="color:#f59e0b"><b>chain</b></span>

In [6]:
bot.show_trace()

## 7. 통계 요약만 간단히

In [7]:
import json
print(json.dumps(bot.summary(), ensure_ascii=False, indent=2))

{
  "total_spans": 7,
  "llm_calls": 3,
  "tokens_in": 104,
  "tokens_out": 56,
  "root_latency_ms": 486.8561250332277
}


## 8. HTML 파일로 저장 — 업무망 반출용

저장된 HTML 은 외부 fetch / `<script src>` / `<link href>` 가 **없으므로** 업무망에서도 파일 하나만으로 열람 가능합니다.

In [8]:
path = bot.save_trace("trace.html")
print("저장됨:", path)

저장됨: trace.html


## 9. 새 thread 로 리셋 — 이전 대화 맥락을 끊기

`bot.reset()` 은 새 `thread_id` 를 발급합니다. `MemorySaver` 는 thread 단위로 상태를 관리하므로 이전 이력과 완전히 분리됩니다. (Tracer는 유지되며, 원하면 `bot.clear_trace()` 로 비울 수 있습니다.)

In [9]:
new_id = bot.reset()
print("new thread_id:", new_id)
print(bot.chat("새 대화 시작! 이전 대화는 기억 안 나겠지?"))

new thread_id: 48450e80
[턴 1] 입력: '새 대화 시작! 이전 대화는 기억 안 나겠지?'. 대화에는 지금까지 1개 메시지가 누적되어 있습니다.


## 10. 사내 실제 LLM 어댑터로 교체하기 (템플릿)

아래 형식을 따르는 어댑터를 주입하면 `MockLLM` 을 교체할 수 있습니다. `self.tracer.span(...)` 블록으로 감싸주면 트레이스 뷰어에 토큰/latency가 함께 표시됩니다.

> 실제 실행은 생략 — 로컬 LLM 로딩 코드는 환경마다 다릅니다.

In [10]:
class MyLocalLLM:
    """사내 실제 LLM 을 감싸는 어댑터 템플릿."""
    def __init__(self, tracer=None):
        self.tracer = tracer
        # self.model = load_local_model(...)  # 실제 로딩 코드로 교체

    def invoke(self, messages):
        if self.tracer is None:
            out, _, _ = self._call(messages)
            return out
        with self.tracer.span("LLM:my-local", kind="llm", inputs=messages) as s:
            out, ti, to = self._call(messages)
            self.tracer.finish(s, outputs=out, tokens_in=ti, tokens_out=to)
            return out

    def _call(self, messages):
        # 실제 추론 호출 자리. 여기서는 동작 확인용 dummy.
        text = "여기에 사내 LLM 응답이 들어갑니다."
        tokens_in = sum(len(str(m.get("content","")).split()) for m in messages)
        tokens_out = len(text.split())
        return {"role": "assistant", "content": text}, tokens_in, tokens_out

# 실제 교체 예시 (주석 처리):
# bot2 = Chatbot(llm=MyLocalLLM())
# print(bot2.chat("안녕!"))
# bot2.show_trace()

print("MyLocalLLM 템플릿 정의 완료. 실제 사용 시 주석을 해제하세요.")

MyLocalLLM 템플릿 정의 완료. 실제 사용 시 주석을 해제하세요.
